# F1 — 모델 완성(동결): 최종 가중치·운영 임계값·게이트웨이 일치·판정 지연

## 왜 하나

모델링은 R4c로 결론이 났다(SSOT 10/15·10/21). 게이트웨이 교체 후보는 **R4c base(DeBERTa-v3-base, 보정 학습 정답, 4토큰)**,
작은 대안은 **R4b V0b(MiniLM)**. 그런데 둘 다 드라이브 용량 때문에 **가중치가 저장되지 않았고**, 운영 임계값·CPU 지연·
게이트웨이 코드와의 점수 일치도 확인한 적이 없다. 이 노트북은 연구가 아니라 **모델을 서비스에 넣을 수 있는 상태로 만드는 마무리**다.

## 무엇을 하나

1. 두 모델을 R4c·R4b와 **같은 방식으로 시드 42 1개씩 재학습**하고, fp16으로 로컬에 먼저 저장한 뒤 드라이브에 복사·크기 검증. 실패해도 마지막 셀에서 zip으로 내려받는다.
2. **재현 확인**: 재학습 모델의 dev-L 성능이 기존 시드 3개와 같은 수준인지.
3. **운영 임계값**: cal-L에서 판정 간격(문장 끝·4·8·16토큰)마다 ARL₀=100 임계값과 등각 보정 서비스 10% 임계값(멀쩡한 응답 표시 ≤ 10%)을 정해 `judge_config.json`에 넣는다.
4. **게이트웨이 일치**: 저장한 가중치를 게이트웨이 `hallugate/judge.py`(그대로 복사)로 불러 같은 점수가 나오는지. 저장소 테스트용 고정 예제 50개도 내보낸다.
5. **판정 지연**: GPU, CPU(스레드 수별) fp32·int8. 사전 규칙으로 배포 구성(모델·재판정 간격)을 고른다.

## 사전 정의 기준 (결과 보기 전, SSOT §6 F1에 고정)

- **F1-1 재현**: 재학습 모델의 dev-L 4토큰 ARL₀=100 엄격 재현율이 원래 정답·C1 모두 **같은 방식 기존 시드 3개 최솟값 − 0.01 이상** (base: R4c, mini: R4b V0b). 미달 모델은 교체 후보에서 뺀다.
- **F1-2 일치**: 게이트웨이 코드로 cal-L 질의 300개 채점 → 노트북 점수와 **Pearson ≥ 0.995, 기본 임계값 판정 일치 ≥ 99%** (GPU·CPU 각각).
- **F1-3 int8**: CPU 동적 양자화 점수가 fp32 대비 **Pearson ≥ 0.99, 판정 일치 ≥ 98%** 면 int8 사용 허용.
- **배포 구성 규칙**: 토큰당 예산 8.5ms(L1 Haiku 평균 속도 환산 4토큰 ≈ 34ms). 각 (모델, 장치·정밀도)에서 **판정 p95 ≤ s × 8.5ms인 최소 재판정 간격 s ∈ {4, 8, 16}**. 우선순위 base(s ≤ 8) → mini(s ≤ 8) → base 문장 끝만.
- **기본 임계값** = cal-L **원래 정답** 등각 보정 서비스 10%(Claude 재판정에 의존하지 않고 보수적). ARL₀=100·C1 임계값도 함께 저장. **임계값·모델 선택에 dev-L을 쓰지 않는다**(재현 확인·보고만).

> **런타임: A100** (L4 가능). base 학습 약 80~95분 + MiniLM 약 10분 + 채점·측정 약 30분 → **약 2~2.5시간**. 끝난 단계는 건너뛴다.
> **실행 전에 드라이브 휴지통을 비울 것.** 드라이브에 `student_vark`, `r0/`, `r4/r4_corrected_labels.json`, `r4b/labels_train.json`·`V0b_s*/scores.npy`, `r4c/base_s*/scores.npy`가 있어야 한다.
> 결과는 드라이브 `selfproject/final/`과 `f1_final_judges.zip`(자동 다운로드).


## 1. 환경 준비

In [ ]:
!pip -q install pysbd sentence-transformers tiktoken
!git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || echo 'RAGTruth already cloned'

import json, time, random, os, gc, copy
import numpy as np, pandas as pd, torch, pysbd, tiktoken
from sklearn.metrics import roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

USE_DRIVE = True
OUT_DIR = '/content/drive/MyDrive/selfproject' if USE_DRIVE else '.'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
STUDENT_DIR = os.path.join(OUT_DIR, 'student_vark')
R0_DIR = os.path.join(OUT_DIR, 'r0')
R1_DIR = FINAL_DIR = os.path.join(OUT_DIR, 'final')
R4C_DIR = os.path.join(OUT_DIR, 'r4c')
R1B_DIR, R1C_DIR, R4_DIR, R4B_DIR = (os.path.join(OUT_DIR, d) for d in ['r1b', 'r1c', 'r4', 'r4b'])
os.makedirs(R1_DIR, exist_ok=True)
for p in [os.path.join(R4B_DIR, 'labels_train.json'), os.path.join(R4_DIR, 'r4_corrected_labels.json')]:
    assert os.path.exists(p), f'{p} 없음'
assert os.path.isdir(STUDENT_DIR), f'{STUDENT_DIR} 없음 (MiniLM 학습 시작점)'
# 드라이브 쓰기 점검: R4c에서는 용량 부족으로 가중치가 조용히 저장되지 않았다
import shutil
_t = os.path.join(FINAL_DIR, '_write_test.bin')
try:
    with open(_t, 'wb') as f: f.write(os.urandom(50 * 2**20))
    ok = os.path.getsize(_t) == 50 * 2**20; os.remove(_t)
except OSError as e:
    ok = False; print('드라이브 쓰기 실패:', e)
print('드라이브 50MB 쓰기', '정상' if ok else '실패 — 휴지통 비우기·용량 확인. 가중치는 마지막 셀의 zip 다운로드로 받는다')
print('device', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')

## 2. 데이터 (R4c와 같음)

In [ ]:
BASE = 'RAGTruth/dataset'
responses = [json.loads(l) for l in open(f'{BASE}/response.jsonl')]
sources   = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{BASE}/source_info.jsonl')}
for r in responses:
    r['task_type'] = sources[r['source_id']]['task_type']

def build_reference(src):
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))

TASKS = ['QA', 'Summary', 'Data2txt']
train_pool = [r for r in responses if r['split'] == 'train']          # test 분할은 사용하지 않음
random.seed(SEED)
dev = []
for tt in TASKS:
    dev += random.sample([r for r in train_pool if r['task_type'] == tt], 120)
dev_ids = {r['id'] for r in dev}
rest = [r for r in train_pool if r['id'] not in dev_ids]
rng = random.Random(11)
teacher_resp = []
for tt in TASKS:
    teacher_resp += rng.sample([r for r in rest if r['task_type'] == tt], 200)
rng.shuffle(teacher_resp)
val = teacher_resp[:int(len(teacher_resp) * 0.15)]                   # student 학습에 쓰지 않은 검증 분할
assert not (dev_ids & {r['id'] for r in val})
print(f'dev {len(dev)} 응답 | 검증 {len(val)} 응답')

In [ ]:
# 학습 응답: S2/S2-b의 gold-large와 같은 구성 (teacher 학습 510 + 추가 2,490). dev·검증과 겹치지 않음을 확인한다.
teacher_train = teacher_resp[len(val):]
used = dev_ids | {r['id'] for r in teacher_resp}
gl_extra = rng.sample([r for r in rest if r['id'] not in used], 3000 - len(teacher_train))
train = teacher_train + gl_extra
tr_ids = {r['id'] for r in train}
assert not (tr_ids & dev_ids) and not (tr_ids & {r['id'] for r in val}), '학습과 dev/검증 겹침'
print(f'학습 {len(train)} 응답 (dev·검증과 겹침 없음)')

In [ ]:
ENC = tiktoken.get_encoding('cl100k_base')
seg = pysbd.Segmenter(language='en', clean=False, char_span=True)

def tokenize_offsets(text):
    ids = ENC.encode(text)
    dec, starts = ENC.decode_with_offsets(ids)
    if dec != text:                       # 드문 경우: 멀티바이트 문자 경계 — 누적 디코드로 대체
        starts, pos = [], 0
        for i in range(len(ids)):
            starts.append(pos); pos = len(ENC.decode(ids[:i + 1]))
    ends = starts[1:] + [len(text)]
    return starts, ends

def prep(r):
    text = r['response']
    ts, te = tokenize_offsets(text)
    # 토큰의 '내용 시작' 위치: cl100k 토큰은 앞 공백을 포함하므로(" The"), 공백을 건너뛴 위치로 문장·환각 구간에 배정한다.
    # (v1에서는 공백 위치로 배정해 문장 경계 토큰이 앞 문장에 붙는 오류가 발생 — 발생 위치의 18%)
    cs = [a + (len(text[a:b]) - len(text[a:b].lstrip())) for a, b in zip(ts, te)]
    cs = [c if c < b else a for c, a, b in zip(cs, ts, te)]          # 공백뿐인 토큰
    spans = [(l['start'], l['end']) for l in (r.get('labels') or [])]
    bad = [j for j in range(len(ts)) if any(min(te[j], b) > max(cs[j], a) for a, b in spans)]
    sents = []
    for sp in seg.segment(text):
        if not sp.sent.strip(): continue
        a, b = sp.start, sp.end
        tok = [j for j in range(len(ts)) if a <= cs[j] < b]
        if not tok: continue
        y = int(any(min(b, e) > max(a, s) for s, e in spans))
        sents.append({'a': a, 'b': b, 't0': tok[0], 't1': tok[-1], 'y': y})
    sent_of = {t: sn['t0'] for sn in sents for t in range(sn['t0'], sn['t1'] + 1)}
    return {'id': str(r['id']), 'task_type': r['task_type'], 'gen_model': r['model'],
            'ref_key': r['source_id'], 'text': text, 'ts': ts, 'te': te, 'n_tok': len(ts),
            'onset': bad[0] if bad else None, 'sents': sents, 'sent_of': sent_of,
            'ends': {sn['t1'] for sn in sents}}

REFS = {}
def to_items(rs):
    out = []
    for r in rs:
        REFS.setdefault(r['source_id'], build_reference(sources[r['source_id']]))
        out.append(prep(r))
    return out

dev_items, val_items = to_items(dev), to_items(val)
for name, it in [('dev', dev_items), ('검증', val_items)]:
    n_h = sum(x['onset'] is not None for x in it)
    print(f'{name}: 응답 {len(it)} (환각 {n_h}, 없음 {len(it)-n_h}) | 토큰 {sum(x["n_tok"] for x in it):,} | '
          f'환각 없는 응답 토큰 {sum(x["n_tok"] for x in it if x["onset"] is None):,}')

## 3. 보정 학습 정답 (R4b Sonnet 재판정, 추가 비용 없음)

In [ ]:
TRAIN_ITEMS = to_items(train)
ITEMS = {f'tr-{x["id"]}': x for x in TRAIN_ITEMS}
LABELS = {k: {int(i): v for i, v in d.items()} for k, d in json.load(open(os.path.join(R4B_DIR, 'labels_train.json'))).items()}
print(f'학습 재판정 {len(LABELS):,}/{len(ITEMS):,}')

In [ ]:
def content_starts(x):
    t = x['text']
    cs = [a + (len(t[a:b]) - len(t[a:b].lstrip())) for a, b in zip(x['ts'], x['te'])]
    return [c if c < b else a for c, a, b in zip(cs, x['ts'], x['te'])]

def locate(x, sn, phrase, cs):
    seg = x['text'][sn['a']:sn['b']]
    ph = (phrase or '').strip()
    k = seg.find(ph) if ph else -1
    if k < 0 and ph: k = seg.lower().find(ph.lower())
    if k < 0: return sn['t0']
    a, b = sn['a'] + k, sn['a'] + k + len(ph)
    for t in range(sn['t0'], sn['t1'] + 1):
        if min(x['te'][t], b) > max(cs[t], a): return t
    return sn['t0']

ADD_ONSET, flip, earlier = {}, 0, 0
for cid, x in ITEMS.items():
    labs = LABELS.get(cid)
    if not labs: continue
    cs, add = None, []
    for i, sn in enumerate(x['sents']):
        if sn['y'] == 0 and labs[i + 1]['label'] == 'H':
            cs = cs or content_starts(x)
            t = locate(x, sn, labs[i + 1]['phrase'], cs); ADD_ONSET[(x['id'], i)] = t; add.append(t)
    if add:
        if x['onset'] is None: flip += 1
        elif min(add) < x['onset']: earlier += 1
n_clean = sum(x['onset'] is None for x in TRAIN_ITEMS); n_hall = len(TRAIN_ITEMS) - n_clean
print(f'추가된 환각 문장 {len(ADD_ONSET):,} | 멀쩡 → 환각 {flip}/{n_clean} ({flip / max(n_clean, 1):.0%}) | 더 이른 환각 {earlier}/{n_hall} ({earlier / max(n_hall, 1):.0%})')

## 4. 학습 판정 지점

In [ ]:
def points(item, s):
    pts = []
    for sn in item['sents']:
        for t in range(sn['t0'], sn['t1'] + 1):
            k = t - sn['t0'] + 1
            if (s is not None and k % s == 0) or t == sn['t1']:
                pts.append((t, item['text'][sn['a']:min(item['te'][t], sn['b'])].strip()))
    return pts


WIN_TOK = 10
train_items = TRAIN_ITEMS
by_tr = {str(r['id']): r for r in train}
TR, P_KEY, P_TXT, P_CAT = [], [], [], []
for x in train_items:
    text = x['text']
    spans = [(l['start'], l['end']) for l in (by_tr[x['id']].get('labels') or [])]
    cs = [a + (len(text[a:b]) - len(text[a:b].lstrip())) if text[a:b].strip() else a for a, b in zip(x['ts'], x['te'])]
    def s_onset(sn):                      # E2와 같다: 이 문장 안에서 환각 구간과 처음 겹치는 토큰
        for t in range(sn['t0'], sn['t1'] + 1):
            if any(min(x['te'][t], b) > max(cs[t], a) for a, b in spans): return t
        return None
    sid = {sn['t0']: i for i, sn in enumerate(x['sents'])}
    son = [s_onset(sn) for sn in x['sents']]
    son = [o if o is not None else ADD_ONSET.get((x['id'], i)) for i, o in enumerate(son)]   # R4b 보정 학습 정답
    groups = {}
    for t, txt in points(x, 2):
        if not txt: continue
        si = sid[x['sent_of'][t]]; o = son[si]
        if o is None: cat = 'neg'
        elif t < o: cat = 'pre'
        elif t <= o + WIN_TOK: cat = 'win'
        else: cat = 'post'
        g = len(P_TXT); P_KEY.append(x['ref_key']); P_TXT.append(txt); P_CAT.append(cat)
        if cat in ('win', 'post'): groups.setdefault(si, []).append(g)
    gl = []
    for si, lst in groups.items():
        win = [g for g in lst if P_CAT[g] == 'win']
        if not win:                       # 창 안에 지점이 없으면 시작 이후 첫 지점을 창으로
            P_CAT[lst[0]] = 'win'; win = [lst[0]]
        gl.append(win)
    TR.append({'groups': gl})
P_CAT = np.array(P_CAT)
print(f'학습 응답 {len(TR)} | 판정 지점 {len(P_TXT):,} | 환각 문장(다중 사례 단위) {sum(len(r["groups"]) for r in TR):,} |',
      pd.Series(P_CAT).value_counts().to_dict())
assert sum(len(r['groups']) for r in TR) > 0

## 5. 근거 검색과 평가용 보정 정답(R4 C1)

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

retriever = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEVICE)
K, MAX_LEN, CHUNK_WORDS, BS = 8, 512, 40, 32

chunk_cache = {}
def get_chunks(key):
    if key not in chunk_cache:
        w = REFS[key].split()
        ch = [' '.join(w[i:i + CHUNK_WORDS]) for i in range(0, max(len(w), 1), CHUNK_WORDS)] or [REFS[key]]
        chunk_cache[key] = (ch, retriever.encode(ch, convert_to_tensor=True, normalize_embeddings=True,
                                                 batch_size=128, show_progress_bar=False))
    return chunk_cache[key]

def contexts(keys, texts, bs=256):
    # 텍스트마다 근거 top-K 조각을 원문 순서로 이어 붙인 문자열
    out = []
    for i in range(0, len(texts), bs):
        embs = retriever.encode(texts[i:i + bs], convert_to_tensor=True, normalize_embeddings=True,
                                batch_size=bs, show_progress_bar=False)
        for key, e in zip(keys[i:i + bs], embs):
            ch, ce = get_chunks(key)
            top = torch.topk(ce @ e, min(K, ce.shape[0])).indices.tolist()
            out.append(' ... '.join(ch[t] for t in sorted(top)))
    return out


# R0 평가 세트(cal-L, dev-L)와 기존 판정기 점수
import pickle
R0P = pickle.load(open(os.path.join(R0_DIR, 'r0_items.pkl'), 'rb'))
Z0 = dict(np.load(os.path.join(R0_DIR, 'r0_scores.npz')))
CAL, DEV = R0P['items']['calL'], R0P['items']['devL']
Q_KEYS, Q_TEXTS = R0P['q_keys'], R0P['q_texts']
for k in set(Q_KEYS):
    REFS.setdefault(k, build_reference(sources[k]))

@torch.no_grad()
def topk_idx(keys, texts, bs=512):
    out = []
    for i in range(0, len(texts), bs):
        embs = retriever.encode(texts[i:i + bs], convert_to_tensor=True, normalize_embeddings=True,
                                batch_size=bs, show_progress_bar=False)
        for key, e in zip(keys[i:i + bs], embs):
            ch, ce = get_chunks(key)
            out.append(sorted(torch.topk(ce @ e, min(K, ce.shape[0])).indices.tolist()))
    return out
ctx_of = lambda key, idx: ' ... '.join(get_chunks(key)[0][j] for j in idx)

t0 = time.perf_counter()
P_IDX = topk_idx(P_KEY, P_TXT)
print(f'학습 지점 근거 검색 {time.perf_counter()-t0:.0f}초')

# 평가 질의: cal-L·dev-L 지점만 (R0와 같은 질의 번호)
EVAL_Q = sorted({q for x in CAL + DEV for _, q in x['pts2']})
t0 = time.perf_counter()
EQ_IDX = dict(zip(EVAL_Q, topk_idx([Q_KEYS[q] for q in EVAL_Q], [Q_TEXTS[q] for q in EVAL_Q])))
print(f'평가 질의 {len(EVAL_Q):,}개 근거 검색 {time.perf_counter()-t0:.0f}초')

# 체크포인트 선택용 cal-sub (cal-L에서 300응답 고정)
CALSUB = random.Random(1).sample(CAL, min(300, len(CAL)))

In [ ]:
# R4 보정 정답 C1 (Opus 재판정)을 cal-L·dev-L에 적용한 사본
CORR = json.load(open(os.path.join(R4_DIR, 'r4_corrected_labels.json')))['C1']
def apply_corr(items, split):
    out = []
    for x in items:
        c = CORR[split].get(x['id'])
        y = copy.copy(x); y['sents'] = [dict(sn) for sn in x['sents']]
        if c is not None:
            y['onset'] = c['onset']
            for i in c['added']: y['sents'][i - 1]['y'] = 1
        out.append(y)
    return out
CAL_C, DEV_C = apply_corr(CAL, 'calL'), apply_corr(DEV, 'devL')
LAB_SETS = {'orig': (CAL, DEV), 'C1': (CAL_C, DEV_C)}
for k, (c, d) in LAB_SETS.items():
    print(f"{k}: cal-L 환각 {sum(x['onset'] is not None for x in c)}, dev-L 환각 {sum(x['onset'] is not None for x in d)}")
# 체크포인트 선택은 보정 정답 기준 cal-sub (같은 300응답)
_byid = {x['id']: x for x in CAL_C}
CALSUB = [_byid[x['id']] for x in CALSUB]

## 6. 평가 함수 (R1b와 같음)

In [ ]:
SC = dict(Z0)

def series(x, scorer):
    m, k = scorer.split('|')
    S_ = SC[m]
    if k == 'end':
        return [(t, S_[q]) for t, q in x['pts2'] if t in x['ends']]
    if k == 'ew2':
        out, s = [], None
        for t, q in x['pts2']:
            v = float(S_[q]); s = v if s is None else 0.5 * v + 0.5 * s
            out.append((t, s))
        return out
    s = int(k)
    return [(t, S_[q]) for t, q in x['pts2'] if t in x['ends'] or (t - x['sent_of'][t] + 1) % s == 0]

In [ ]:
def calibrate(items, scorer, arl0):
    clean = [x for x in items if x['onset'] is None]
    n_tok = sum(x['n_tok'] for x in clean)
    sc = np.sort(np.concatenate([[s for _, s in series(x, scorer)] for x in clean]))[::-1]
    n_alarm = max(1, int(n_tok / arl0))
    return float(sc[n_alarm - 1]) + 1e-6 if n_alarm <= len(sc) else float(sc[-1])

def evaluate(items, scorer, thr):
    # 엄격 기준(주 지표, 선행 연구와 같은 정의): 응답의 **첫 경보**가 발생 θ 이후여야 탐지로 인정.
    #   θ 전에 경보가 울리면 그 응답은 오경보로 끝난 것(미탐지)으로 본다.
    # 관대 기준(참고): θ 전 경보는 무시하고 θ 이후 첫 경보를 탐지로 인정 — v1의 주 지표였으나 재현율이 부풀려짐.
    clean = [x for x in items if x['onset'] is None]
    hall = [x for x in items if x['onset'] is not None]
    alarms = sum(sum(s >= thr for _, s in series(x, scorer)) for x in clean)
    arl = sum(x['n_tok'] for x in clean) / max(alarms, 1)
    clean_flag = np.mean([any(s >= thr for _, s in series(x, scorer)) for x in clean])
    st_det, st_d, st_ch, cedd, le_det, pre = 0, [], [], [], 0, 0
    for x in hall:
        ser = series(x, scorer)
        first = next((t for t, s in ser if s >= thr), None)
        le_hit = next((t for t, s in ser if t >= x['onset'] and s >= thr), None)
        le_det += le_hit is not None
        pre += first is not None and first < x['onset']
        if first is not None and first >= x['onset']:
            st_det += 1; st_d.append(first - x['onset']); cedd.append(first - x['onset'])
            st_ch.append(x['te'][first] - x['ts'][x['onset']])
        else:
            cedd.append(x['n_tok'] - x['onset'])
    return {'realized_ARL0_dev': arl, 'clean_resp_flagged': clean_flag,
            'recall': st_det / len(hall), 'delay_tok_mean': np.mean(st_d) if st_d else np.nan,
            'delay_tok_median': np.median(st_d) if st_d else np.nan,
            'delay_chars_median': np.median(st_ch) if st_ch else np.nan,
            'censored_EDD_tok': np.mean(cedd), 'pre_onset_alarm_rate': pre / len(hall),
            'recall_lenient': le_det / len(hall),
            'calls_per_resp': np.mean([len(series(x, scorer)) for x in items])}

def calibrate_flag(items, scorer, rate):
    # 서비스 기준 보정: 환각 없는 응답 중 표시되는 비율이 rate 이하가 되는 임계값 (응답별 최대 점수의 분위수)
    mx = np.array([max([s for _, s in series(x, scorer)] or [-1e9]) for x in items if x['onset'] is None])
    return float(np.quantile(mx, 1 - rate)) + 1e-6

In [ ]:
def resp_auc(items, m):
    y = [int(x['onset'] is not None) for x in items]
    s = [max([SC[m][q] for _, q in x['pts2']] or [-1e9]) for x in items]
    return roc_auc_score(y, s)

def sent_auc(items, m):
    ys, ss = [], []
    for x in items:
        d = dict(x['pts2'])
        for sn in x['sents']:
            if sn['t1'] in d: ys.append(sn['y']); ss.append(SC[m][d[sn['t1']]])
    return roc_auc_score(ys, ss)

def per_resp(items, sc, thr):
    d, l = [], []
    for x in items:
        if x['onset'] is None: continue
        first = next((t for t, s in series(x, sc) if s >= thr), None)
        ok = first is not None and first >= x['onset']
        d.append(float(ok)); l.append(first - x['onset'] if ok else np.nan)
    return np.array(d), np.array(l)

def boot(a, b, n=1000, seed=0):
    da, la = per_resp(DEV, a, calibrate(CAL, a, 100)); db, lb = per_resp(DEV, b, calibrate(CAL, b, 100))
    rng = np.random.default_rng(seed); N = len(da); rd = []
    for _ in range(n):
        i = rng.integers(0, N, N); rd.append(da[i].mean() - db[i].mean())
    lo, hi = np.percentile(rd, [2.5, 97.5])
    return {'A': a, 'B': b, 'recall_A': da.mean(), 'recall_B': db.mean(), 'diff': da.mean() - db.mean(), 'lo': lo, 'hi': hi}

# 기준선 확인 (R0 dev-L과 같아야 함)
for sc, ref in [('B|end', 0.5007), ('B|4', 0.4809), ('E2|4', 0.4506)]:
    got = evaluate(DEV, sc, calibrate(CAL, sc, 100))['recall']
    print(f'{sc:6} R0 {ref:.4f} | 지금 {got:.4f} | {"일치" if abs(got-ref) <= 0.002 else "불일치 — 확인 필요"}')

## 7. 학습 루프 — base는 R1c(=R4c), MiniLM은 R1b(=R4b) 그대로

In [ ]:
KEEP_WEIGHTS = {42}
from transformers import get_linear_schedule_with_warmup
import torch.nn.functional as F, shutil

BACKBONES = {'mini': ('cross-encoder/nli-MiniLM2-L6-H768', 2e-5),
             'base': ('cross-encoder/nli-deberta-v3-base', 2e-5),
             'large': ('lytang/MiniCheck-DeBERTa-v3-Large', 1e-5)}
BATCH_SEQ, MICRO, N_EVALS = 64, 16, 4
BF16 = DEVICE == 'cuda' and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16
if not BF16: print('⚠️ bf16 미지원 GPU — DeBERTa는 fp16에서 불안정할 수 있음 (A100/L4 권장)')

def enc_ids(tok, idxs):
    return tok([ctx_of(P_KEY[g], P_IDX[g]) for g in idxs], [P_TXT[g] for g in idxs], truncation='only_first',
               max_length=MAX_LEN, padding=True, return_tensors='pt').to(DEVICE)

@torch.no_grad()
def score_queries(tok, model, qs, bs=128):
    model.eval(); out = np.zeros(len(qs), dtype=np.float32)
    order = sorted(range(len(qs)), key=lambda i: len(Q_TEXTS[qs[i]]))
    for i in range(0, len(order), bs):
        b = [qs[j] for j in order[i:i + bs]]
        enc = tok([ctx_of(Q_KEYS[q], EQ_IDX[q]) for q in b], [Q_TEXTS[q] for q in b], truncation='only_first',
                  max_length=MAX_LEN, padding=True, return_tensors='pt').to(DEVICE)
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
            out[order[i:i + bs]] = model(**enc).logits[:, 0].float().cpu().numpy()
    model.train()
    return out

CALSUB_Q = sorted({q for x in CALSUB for _, q in x['pts2']})
def calsub_metric(tok, model):
    v = score_queries(tok, model, CALSUB_Q)
    SC['_cur'] = np.zeros(len(Q_KEYS), dtype=np.float32); SC['_cur'][CALSUB_Q] = v
    r4 = evaluate(CALSUB, '_cur|4', calibrate(CALSUB, '_cur|4', 100))
    r2 = evaluate(CALSUB, '_cur|2', calibrate(CALSUB, '_cur|2', 100))
    d4 = r4['delay_tok_median']
    return float((r4['recall'] + r2['recall']) / 2), float(d4) if d4 == d4 else 1e9

Y_ALL = np.isin(P_CAT, ['win', 'post']).astype(np.float32)          # V0b: 문장 단위 시작 이후 = 1

def load_init(name):
    path, _ = BACKBONES[name]
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSequenceClassification.from_pretrained(path, num_labels=1, ignore_mismatched_sizes=True).to(DEVICE)
    return tok, model

def train_backbone(name, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tok, model = load_init(name)
    lr = BACKBONES[name][1]
    order = list(range(len(P_TXT))); random.Random(seed).shuffle(order)
    n_steps = max(1, len(order) // BATCH_SEQ)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * n_steps), n_steps)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda' and not BF16))
    best, d0 = calsub_metric(tok, model)
    best_key, best_state, hist = (d0 <= 9, best), copy.deepcopy(model.state_dict()), [(0, best, d0)]
    print(f'  [{name} s{seed}] 시작 cal-sub 재현율 {best:.3f}, 지연 {d0} | {n_steps}단계, 학습률 {lr}')
    eval_at = {int(n_steps * (i + 1) / N_EVALS) for i in range(N_EVALS)}
    model.train(); t0 = time.perf_counter()
    for step in range(1, n_steps + 1):
        batch = order[(step - 1) * BATCH_SEQ: step * BATCH_SEQ]
        opt.zero_grad(); tot = 0.0
        for k in range(0, len(batch), MICRO):
            mb = batch[k:k + MICRO]
            with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
                z = model(**enc_ids(tok, mb)).logits[:, 0].float()
            loss = F.binary_cross_entropy_with_logits(z, torch.tensor(Y_ALL[mb], device=DEVICE), reduction='sum') / len(batch)
            scaler.scale(loss).backward(); tot += loss.item()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sch.step()
        if step in eval_at:
            va, dv = calsub_metric(tok, model); hist.append((step, va, dv))
            if (dv <= 9, va) > best_key: best_key, best, best_state = (dv <= 9, va), va, copy.deepcopy(model.state_dict())
            print(f'  [{name} s{seed}] 단계 {step}/{n_steps} | 손실 {tot:.3f} | cal-sub 재현율 {va:.3f}, 지연 {dv} | {(time.perf_counter()-t0)/60:.1f}분')
    model.load_state_dict(best_state)
    return tok, model, hist

def run(name, seed):
    out = os.path.join(R1_DIR, f'{name}_s{seed}')
    sp = os.path.join(out, 'scores.npy')
    if not os.path.exists(sp):
        has_w = any(os.path.exists(os.path.join(out, f)) for f in ['model.safetensors', 'pytorch_model.bin'])
        if os.path.exists(os.path.join(out, 'config.json')) and not has_w:
            print(f'  [{name} s{seed}] 저장이 끊긴 폴더 → 지우고 다시 학습'); shutil.rmtree(out)
        if has_w:
            tok = AutoTokenizer.from_pretrained(out); model = AutoModelForSequenceClassification.from_pretrained(out).to(DEVICE)
        else:
            tok, model, hist = train_backbone(name, seed)
            os.makedirs(out, exist_ok=True); tok.save_pretrained(out); model.save_pretrained(out)
            pd.DataFrame(hist, columns=['step', 'calsub_recall', 'calsub_delay4']).to_csv(os.path.join(out, 'hist.csv'), index=False)
        t0 = time.perf_counter()
        v = score_queries(tok, model, EVAL_Q)
        full = np.full(len(Q_KEYS), np.nan, dtype=np.float32); full[EVAL_Q] = v
        np.save(sp, full)
        if seed not in KEEP_WEIGHTS:                     # 드라이브 용량: 가중치는 시드 42만 남긴다
            for f in ['model.safetensors', 'pytorch_model.bin']:
                if os.path.exists(os.path.join(out, f)): os.remove(os.path.join(out, f))
        print(f'  [{name} s{seed}] 평가 질의 채점 {time.perf_counter()-t0:.0f}초')
        del model; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
    SC[f'{name}_s{seed}'] = np.load(sp)
    print(f'[{name} s{seed}] 준비 완료')

In [ ]:
from transformers import get_linear_schedule_with_warmup
import torch.nn.functional as F

BATCH_SEQ, LR, POST_W, GROUP_W, MARGIN, MINE_M, N_EVALS = 64, 2e-5, 0.2, 5.0, 1.0, 4, 4
BF16 = DEVICE == 'cuda' and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16

def encode(tok, idxs):
    ctx = [ctx_of(P_KEY[g], P_IDX[g]) for g in idxs]
    return tok(ctx, [P_TXT[g] for g in idxs], truncation='only_first', max_length=MAX_LEN,
               padding=True, return_tensors='pt').to(DEVICE)

def make_units(variant, rng):
    units = []
    for g in range(len(P_TXT)):
        c = P_CAT[g]
        if variant == 'V0b':
            units.append(('s', g, 1.0 if c in ('win', 'post') else 0.0, 1.0))
        elif c == 'post':
            units.append(('s', g, 1.0, POST_W))
        elif c in ('neg', 'pre'):
            units.append(('s', g, 0.0, 1.0))
    if variant == 'V1b':
        for r in TR:
            for win in r['groups']:
                units.append(('g', win))
    rng.shuffle(units)
    return units

@torch.no_grad()
def score_ids(tok, model, idxs, bs=256):
    model.eval(); out = []
    for i in range(0, len(idxs), bs):
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
            out.append(model(**encode(tok, idxs[i:i + bs])).logits.squeeze(-1).float().cpu())
    model.train()
    return torch.cat(out).numpy() if out else np.zeros(0)

@torch.no_grad()
def score_queries(tok, model, qs, bs=256):
    model.eval(); out = np.zeros(len(qs), dtype=np.float32)
    order = sorted(range(len(qs)), key=lambda i: len(Q_TEXTS[qs[i]]))
    for i in range(0, len(order), bs):
        b = [qs[j] for j in order[i:i + bs]]
        ctx = [ctx_of(Q_KEYS[q], EQ_IDX[q]) for q in b]
        enc = tok(ctx, [Q_TEXTS[q] for q in b], truncation='only_first', max_length=MAX_LEN,
                  padding=True, return_tensors='pt').to(DEVICE)
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
            out[order[i:i + bs]] = model(**enc).logits.squeeze(-1).float().cpu().numpy()
    model.train()
    return out

CALSUB_Q = sorted({q for x in CALSUB for _, q in x['pts2']})
def calsub_metric(tok, model):
    v = score_queries(tok, model, CALSUB_Q)
    SC['_cur'] = np.zeros(len(Q_KEYS), dtype=np.float32); SC['_cur'][CALSUB_Q] = v
    r4 = evaluate(CALSUB, '_cur|4', calibrate(CALSUB, '_cur|4', 100))
    r2 = evaluate(CALSUB, '_cur|2', calibrate(CALSUB, '_cur|2', 100))
    d4 = r4['delay_tok_median']
    return float((r4['recall'] + r2['recall']) / 2), float(d4) if d4 == d4 else 1e9

def group_loss(variant, win, tok, model):
    return win, []

def train_variant(variant, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tok = AutoTokenizer.from_pretrained(STUDENT_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(STUDENT_DIR).to(DEVICE)
    units = make_units(variant, random.Random(seed))
    n_seq = sum(1 if u[0] == 's' else 6 for u in units)
    n_steps = max(1, n_seq // BATCH_SEQ)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * n_steps), n_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda' and not BF16))
    best, d0 = calsub_metric(tok, model)
    best_key, best_state, hist = (d0 <= 9, best), copy.deepcopy(model.state_dict()), [(0, best, d0)]
    print(f'  [{variant} s{seed}] 시작 cal-sub 재현율 {best:.3f}, 지연 {d0} | 예상 {n_steps}단계')
    eval_at = {int(n_steps * (i + 1) / N_EVALS) for i in range(N_EVALS)}
    model.train(); step, ui, t0 = 0, 0, time.perf_counter()
    while ui < len(units):
        singles, groups, n = [], [], 0
        while ui < len(units) and n < BATCH_SEQ:
            u = units[ui]; ui += 1
            if u[0] == 's':
                singles.append(u); n += 1
            else:
                win, hard = group_loss(variant, u[1], tok, model)
                if win:
                    groups.append((win, hard)); n += len(win) + len(hard)
        ids = [u[1] for u in singles] + [g for w, h in groups for g in w + h]
        if not ids: continue
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
            z = model(**encode(tok, ids)).logits.squeeze(-1).float()
        ns = len(singles)
        y = torch.tensor([u[2] for u in singles], device=DEVICE); w = torch.tensor([u[3] for u in singles], device=DEVICE)
        loss_num = (F.binary_cross_entropy_with_logits(z[:ns], y, reduction='none') * w).sum() if ns else z.sum() * 0
        den = w.sum() if ns else torch.tensor(0.0, device=DEVICE)
        pos = ns
        for win, hard in groups:
            zw = z[pos:pos + len(win)]; zh = z[pos + len(win):pos + len(win) + len(hard)]; pos += len(win) + len(hard)
            s_no = F.logsigmoid(-zw).sum().clamp(max=-1e-6)              # 창 안에서 한 번도 안 울릴 로그확률
            l = -torch.log(-torch.expm1(s_no))                            # 적어도 하나 울릴 확률의 음의 로그
            if len(hard):
                l = l + F.softplus(torch.logsumexp(zh, 0) - torch.logsumexp(zw, 0) + MARGIN)
            loss_num = loss_num + GROUP_W * l; den = den + GROUP_W
        loss = loss_num / den.clamp(min=1e-6)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step()
        step += 1
        if step in eval_at or ui >= len(units):
            va, dv = calsub_metric(tok, model); hist.append((step, va, dv))
            if (dv <= 9, va) > best_key: best_key, best, best_state = (dv <= 9, va), va, copy.deepcopy(model.state_dict())
            print(f'  [{variant} s{seed}] 단계 {step} | 손실 {loss.item():.3f} | cal-sub 재현율 {va:.3f}, 지연 {dv} | {(time.perf_counter()-t0)/60:.1f}분')
    model.load_state_dict(best_state)
    return tok, model, best, hist

import shutil
def run(variant, seed):
    out = os.path.join(R1_DIR, f'{variant}_s{seed}')
    sp = os.path.join(out, 'scores.npy')
    if not os.path.exists(sp):
        has_w = any(os.path.exists(os.path.join(out, f)) for f in ['model.safetensors', 'pytorch_model.bin'])
        if os.path.exists(os.path.join(out, 'config.json')) and not has_w:
            print(f'  [{variant} s{seed}] 저장이 끊긴 폴더(가중치 없음) → 지우고 다시 학습'); shutil.rmtree(out)
        if has_w:
            tok = AutoTokenizer.from_pretrained(out); model = AutoModelForSequenceClassification.from_pretrained(out).to(DEVICE)
        else:
            tok, model, best, hist = train_variant(variant, seed)
            os.makedirs(out, exist_ok=True); tok.save_pretrained(out); model.save_pretrained(out)
            pd.DataFrame(hist, columns=['step', 'calsub_recall', 'calsub_delay4']).to_csv(os.path.join(out, 'hist.csv'), index=False)
        t0 = time.perf_counter()
        v = score_queries(tok, model, EVAL_Q)
        full = np.full(len(Q_KEYS), np.nan, dtype=np.float32); full[EVAL_Q] = v
        np.save(sp, full)
        print(f'  [{variant} s{seed}] 평가 질의 채점 {time.perf_counter()-t0:.0f}초')
        del model; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
    SC[f'{variant}_s{seed}'] = np.load(sp)
    print(f'[{variant} s{seed}] 준비 완료')

## 8. 최종 모델 학습·저장 (시드 42)

로컬(`/content/final_local`)에 fp16으로 먼저 저장하고 드라이브에 복사한 뒤 파일 크기가 같은지 확인한다.
평가 점수는 **저장한 파일을 다시 불러서** 매긴다(임계값이 실제로 배포할 가중치의 점수와 맞도록).

In [ ]:
LOCAL = '/content/final_local'
os.makedirs(LOCAL, exist_ok=True)
WFILES = ['model.safetensors', 'pytorch_model.bin']
NAMES = ['base', 'mini']
BACKBONE_OF = {'base': 'cross-encoder/nli-deberta-v3-base (R1c 방식, 사전학습 NLI에서 시작)',
               'mini': 'cross-encoder/nli-MiniLM2-L6-H768 → student_vark에서 시작 (R1b V0b 방식)'}
DRIVE_OK = {}

def has_weights(d):
    return os.path.isdir(d) and any(os.path.exists(os.path.join(d, f)) and os.path.getsize(os.path.join(d, f)) > 0 for f in WFILES)

def sizes(d):
    return {f: os.path.getsize(os.path.join(d, f)) for f in sorted(os.listdir(d)) if os.path.isfile(os.path.join(d, f))}

def to_drive(name):
    src, dst = os.path.join(LOCAL, f'judge_{name}'), os.path.join(FINAL_DIR, f'judge_{name}')
    try:
        shutil.copytree(src, dst, dirs_exist_ok=True)
        ok = sizes(src) == {f: v for f, v in sizes(dst).items() if f in sizes(src)}
    except OSError as e:
        print(f'  [{name}] 드라이브 복사 실패: {e}'); ok = False
    DRIVE_OK[name] = ok
    print(f'  [{name}] 드라이브 저장 {"확인" if ok else "실패 — 마지막 셀 zip을 꼭 내려받을 것"} ({sum(sizes(src).values()) / 2**20:.0f}MB)')

def load_judge(name):
    d = os.path.join(LOCAL, f'judge_{name}')
    tok = AutoTokenizer.from_pretrained(d)
    model = AutoModelForSequenceClassification.from_pretrained(d).float().to(DEVICE)   # fp16 저장 → fp32로 올림 (게이트웨이와 같게)
    return tok, model

def finalize(name):
    d, dd = os.path.join(LOCAL, f'judge_{name}'), os.path.join(FINAL_DIR, f'judge_{name}')
    if not has_weights(d) and has_weights(dd):
        shutil.copytree(dd, d, dirs_exist_ok=True); print(f'  [{name}] 드라이브 저장본 사용')
    if not has_weights(d):
        t0 = time.perf_counter()
        if name == 'base':
            tok, model, hist = train_backbone('base', 42)
        else:
            tok, model, _, hist = train_variant('V0b', 42)
        os.makedirs(d, exist_ok=True)
        tok.save_pretrained(d); model.half().save_pretrained(d)
        pd.DataFrame(hist, columns=['step', 'calsub_recall', 'calsub_delay4']).to_csv(os.path.join(d, 'hist.csv'), index=False)
        print(f'  [{name}] 학습 {(time.perf_counter() - t0) / 60:.0f}분, 로컬 저장')
        del model; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
        to_drive(name)
    sp, spd = os.path.join(LOCAL, f'scores_{name}.npy'), os.path.join(FINAL_DIR, f'scores_{name}.npy')
    if not os.path.exists(sp) and os.path.exists(spd): shutil.copy(spd, sp)
    if not os.path.exists(sp):
        tok, model = load_judge(name)
        t0 = time.perf_counter()
        v = score_queries(tok, model, EVAL_Q)
        full = np.full(len(Q_KEYS), np.nan, dtype=np.float32); full[EVAL_Q] = v
        np.save(sp, full)
        try: shutil.copy(sp, spd)
        except OSError as e: print('  점수 드라이브 저장 실패', e)
        print(f'  [{name}] 평가 질의 {len(EVAL_Q):,}개 채점 {time.perf_counter() - t0:.0f}초')
        del model; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
    SC[f'F_{name}'] = np.load(sp)
    DRIVE_OK.setdefault(name, has_weights(dd))
    print(f'[{name}] 준비 완료')

for name in ['mini', 'base']:          # 짧은 MiniLM 먼저 (저장 경로 점검)
    finalize(name)

## 9. F1-1 재현 확인 (dev-L, 기존 시드 3개와 비교)

In [ ]:
SEEDS = [42, 7, 2026]
REF_PATHS = {'base': [os.path.join(R4C_DIR, f'base_s{s}', 'scores.npy') for s in SEEDS],
             'mini': [os.path.join(R4B_DIR, f'V0b_s{s}', 'scores.npy') for s in SEEDS]}
for name, paths in REF_PATHS.items():
    for s, p in zip(SEEDS, paths):
        assert os.path.exists(p), f'{p} 없음'
        SC[f'ref_{name}_s{s}'] = np.load(p)

F1_1, rep = {}, []
for name in NAMES:
    ok = True
    for lab, (cal_, dev_) in LAB_SETS.items():
        sc = f'F_{name}|4'
        r = evaluate(dev_, sc, calibrate(cal_, sc, 100))
        refs = [evaluate(dev_, f'ref_{name}_s{s}|4', calibrate(cal_, f'ref_{name}_s{s}|4', 100))['recall'] for s in SEEDS]
        floor = min(refs) - 0.01; ok &= r['recall'] >= floor
        rep.append({'model': name, 'labels': lab, 'recall': r['recall'], 'delay_med': r['delay_tok_median'],
                    'ref_seeds': np.round(refs, 3).tolist(), 'floor': floor, 'pass': r['recall'] >= floor})
    F1_1[name] = bool(ok)
rep = pd.DataFrame(rep)
print(rep.round(3).to_string(index=False))
for name in NAMES:
    print(f"=== F1-1 재현 [{name}]: {'충족' if F1_1[name] else '미달 → 교체 후보에서 제외'} ===")

## 10. 운영 임계값 (cal-L에서만 정함)과 dev-L 확인

- `arl100_*`: 멀쩡한 응답 100토큰당 경보 1회(엄격 평가 조건)
- `svc10_*`: **등각 보정** — 멀쩡한 응답 중 표시 비율 ≤ 10% (응답별 최대 점수의 ⌈(n+1)(1−α)⌉번째 값)
- `*_orig`: 원래 RAGTruth 정답의 멀쩡한 응답으로 보정(기본), `*_C1`: Opus 보정 정답 기준

In [ ]:
KINDS_F = ['end', '4', '8', '16']
ALPHA = 0.10
def conformal_flag(items, scorer, alpha=ALPHA):
    mx = np.sort([max([s for _, s in series(x, scorer)] or [-1e9]) for x in items if x['onset'] is None])
    n = len(mx); k = int(np.ceil((n + 1) * (1 - alpha)))
    return float(mx[k - 1]) + 1e-6 if k <= n else float('inf')

THR = {}
for name in NAMES:
    THR[name] = {}
    for k in KINDS_F:
        sc = f'F_{name}|{k}'
        THR[name][k] = {}
        for lab, (cal_, _) in LAB_SETS.items():
            THR[name][k][f'arl100_{lab}'] = calibrate(cal_, sc, 100)
            THR[name][k][f'svc10_{lab}'] = conformal_flag(cal_, sc)
DEFAULT = {'kind': '4', 'threshold': 'svc10_orig'}

rows = []
for name in NAMES:
    for k in KINDS_F:
        sc = f'F_{name}|{k}'
        for tk in ['arl100_orig', 'svc10_orig', 'svc10_C1']:
            for evlab, (_, dev_) in LAB_SETS.items():
                r = evaluate(dev_, sc, THR[name][k][tk])
                rows.append({'model': name, 'kind': k, 'thr': tk, 'eval_labels': evlab, 'threshold': THR[name][k][tk],
                             'recall': r['recall'], 'delay_med': r['delay_tok_median'], 'pre_onset': r['pre_onset_alarm_rate'],
                             'clean_flagged': r['clean_resp_flagged'], 'calls_per_resp': r['calls_per_resp']})
dev_tab = pd.DataFrame(rows)
pd.set_option('display.width', 250)
print(dev_tab.round(3).to_string(index=False))
# 기준선 참고 (같은 방식, dev-L)
for lab, (cal_, dev_) in LAB_SETS.items():
    r = evaluate(dev_, 'B|end', calibrate(cal_, 'B|end', 100))
    print(f"(참고) B 문장 끝 ARL₀=100 {lab}: 재현율 {r['recall']:.3f}, 지연 {r['delay_tok_median']:.0f}")

## 11. F1-2 게이트웨이 코드와 점수 일치

저장소의 `hallugate/judge.py`를 그대로 써서(노트북 생성 시점의 코드) 저장한 가중치를 불러 채점한다.
저장소 테스트용 고정 예제 50개(근거, 판정 텍스트, CPU fp32 점수)를 `parity_fixture.json`으로 내보낸다.

In [ ]:
JUDGE_SRC = '"""문장 판정기.\n\nS2/S2-b와 같은 방식:\n  근거 문서 → 40단어 조각 → 조각 임베딩 (요청당 1회, prepare)\n  문장 → 임베딩 → 유사한 조각 top-k (원문 순서로 정렬) → 교차 인코더 → logit (클수록 환각)\n"""\nimport threading\nimport time\nfrom dataclasses import dataclass\n\nimport torch\n\n\n@dataclass\nclass Verdict:\n    score: float          # logit, 클수록 환각\n    hallucinated: bool\n    judge_ms: float       # 검색 + 판정 시간\n\n\n@dataclass\nclass RefIndex:\n    chunks: list\n    emb: object           # torch.Tensor [n_chunks, d]\n    prepare_ms: float\n\n\ndef chunk_text(text, n):\n    w = text.split()\n    return [\' \'.join(w[i:i + n]) for i in range(0, max(len(w), 1), n)] or [text]\n\n\nclass StudentJudge:\n    def __init__(self, model_dir, embedder=\'sentence-transformers/all-MiniLM-L6-v2\', k=8, max_len=512,\n                 chunk_words=40, threshold=0.0, device=\'\'):\n        from transformers import AutoModelForSequenceClassification, AutoTokenizer\n        self.device = device or (\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        self.tok = AutoTokenizer.from_pretrained(model_dir)\n        # 가중치가 fp16으로 저장돼 있어도 fp32로 올린다 (CPU 판정, 노트북 점수와 일치). GPU에서는 autocast가 반정밀도로 돌린다.\n        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir).float().to(self.device).eval()\n        if isinstance(embedder, str):\n            from sentence_transformers import SentenceTransformer\n            embedder = SentenceTransformer(embedder, device=self.device)\n        self.embedder = embedder\n        self.k, self.max_len, self.chunk_words, self.threshold = k, max_len, chunk_words, threshold\n        self._lock = threading.Lock()     # 모델 1개를 여러 요청이 공유하므로 직렬화\n\n    def _embed(self, texts):\n        return self.embedder.encode(texts, convert_to_tensor=True, normalize_embeddings=True,\n                                    show_progress_bar=False)\n\n    def prepare(self, reference):\n        t0 = time.perf_counter()\n        chunks = chunk_text(reference, self.chunk_words)\n        with self._lock:\n            emb = self._embed(chunks)\n        return RefIndex(chunks, emb, (time.perf_counter() - t0) * 1000)\n\n    @torch.no_grad()\n    def judge(self, ref, sentence):\n        t0 = time.perf_counter()\n        with self._lock:\n            e = self._embed([sentence])[0]\n            top = torch.topk(ref.emb @ e, min(self.k, ref.emb.shape[0])).indices.tolist()\n            ctx = \' ... \'.join(ref.chunks[i] for i in sorted(top))\n            enc = self.tok([ctx], [sentence], truncation=\'only_first\', max_length=self.max_len,\n                           return_tensors=\'pt\').to(self.device)\n            with torch.autocast(self.device, enabled=(self.device == \'cuda\')):\n                score = float(self.model(**enc).logits.squeeze().float().cpu())\n            if self.device == \'cuda\':\n                torch.cuda.synchronize()\n        return Verdict(score, score >= self.threshold, (time.perf_counter() - t0) * 1000)\n\n\nclass LexicalJudge:\n    """모델 없이 파이프라인을 시험하기 위한 대체 판정기 (테스트·데모 전용).\n    근거에 없는 단어 비율을 점수로 쓴다. 판별력은 없다고 봐야 한다."""\n\n    def __init__(self, threshold=0.5, delay_ms=0.0):\n        self.threshold, self.delay_ms = threshold, delay_ms\n\n    def prepare(self, reference):\n        return RefIndex([reference], {w.lower().strip(\'.,\') for w in reference.split()}, 0.0)\n\n    def judge(self, ref, sentence):\n        t0 = time.perf_counter()\n        if self.delay_ms:\n            time.sleep(self.delay_ms / 1000)\n        words = [w.lower().strip(\'.,\') for w in sentence.split() if w.strip(\'.,\')]\n        score = sum(w not in ref.emb for w in words) / max(len(words), 1)\n        return Verdict(score, score >= self.threshold, (time.perf_counter() - t0) * 1000)\n'
open('hallugate_judge.py', 'w').write(JUDGE_SRC)
import importlib, sys
sys.path.insert(0, '.')
import hallugate_judge; importlib.reload(hallugate_judge)
from hallugate_judge import StudentJudge

CAL_Q = sorted({q for x in CAL for _, q in x['pts2']})
Q300 = random.Random(2027).sample(CAL_Q, 300)

def gw_scores(j, qs):
    cache, out = {}, []
    for q in qs:
        key = Q_KEYS[q]
        if key not in cache: cache[key] = j.prepare(REFS[key])
        out.append(j.judge(cache[key], Q_TEXTS[q]).score)
    return np.array(out, dtype=np.float64)

def agree(a, b, thr):
    return float(np.mean((a >= thr) == (b >= thr)))

F1_2, PAR, GW_CPU = {}, [], {}
for name in NAMES:
    d = os.path.join(LOCAL, f'judge_{name}')
    nb = SC[f'F_{name}'][Q300].astype(np.float64)
    thr = THR[name][DEFAULT['kind']][DEFAULT['threshold']]
    ok = True
    for dev in (['cuda'] if DEVICE == 'cuda' else []) + ['cpu']:
        torch.set_num_threads(os.cpu_count())
        j = StudentJudge(d, k=K, max_len=MAX_LEN, chunk_words=CHUNK_WORDS, device=dev)
        g = gw_scores(j, Q300)
        if dev == 'cpu': GW_CPU[name] = g
        pr, ag, mx = float(np.corrcoef(nb, g)[0, 1]), agree(nb, g, thr), float(np.max(np.abs(nb - g)))
        nan = int(np.isnan(g).sum())
        okd = nan == 0 and pr >= 0.995 and ag >= 0.99; ok &= okd
        PAR.append({'model': name, 'device': dev, 'pearson': pr, 'agree': ag, 'max_abs_diff': mx, 'nan': nan, 'pass': okd})
        del j; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
    F1_2[name] = bool(ok)
    fx = [{'reference': REFS[Q_KEYS[q]], 'text': Q_TEXTS[q], 'score_cpu_fp32': float(s)} for q, s in zip(Q300[:50], GW_CPU[name][:50])]
    json.dump({'model': f'judge_{name}', 'k': K, 'max_len': MAX_LEN, 'chunk_words': CHUNK_WORDS,
               'embedder': 'sentence-transformers/all-MiniLM-L6-v2', 'items': fx},
              open(os.path.join(d, 'parity_fixture.json'), 'w'), ensure_ascii=False)
PAR = pd.DataFrame(PAR)
print(PAR.round(4).to_string(index=False))
for name in NAMES:
    print(f"=== F1-2 게이트웨이 일치 [{name}]: {'충족' if F1_2[name] else '미달 → 게이트웨이 판정기 코드 수정 필요'} ===")

## 12. F1-3 int8과 판정 지연, 배포 구성

판정 1회 = 판정 텍스트 임베딩 + 근거 top-8 검색 + 교차 인코더(게이트웨이 `judge()`와 같음). 근거 준비(`prepare`)는 요청당 1회라 따로 적는다.
CPU는 이 런타임의 vCPU로 잰다 — 실제 배포 서버와 다를 수 있으므로 **스레드 수별로** 적는다.

In [ ]:
TEXTS = [(REFS[Q_KEYS[q]], Q_TEXTS[q]) for q in Q300]
def bench(j, n):
    refs = {}
    for r, _ in TEXTS[:5]:                             # 예열
        if r not in refs: refs[r] = j.prepare(r)
        j.judge(refs[r], 'warm up sentence.')
    prep, ms = [], []
    for r, t in TEXTS[:n]:
        if r not in refs:
            refs[r] = j.prepare(r); prep.append(refs[r].prepare_ms)
        ms.append(j.judge(refs[r], t).judge_ms)
    return np.percentile(ms, 50), np.percentile(ms, 95), (np.median(prep) if prep else np.nan)

def quantize(j):
    j.model = torch.ao.quantization.quantize_dynamic(j.model, {torch.nn.Linear}, dtype=torch.qint8)
    return j

F1_3, I8 = {}, []
NT = os.cpu_count()
torch.set_num_threads(NT)
for name in NAMES:
    j = quantize(StudentJudge(os.path.join(LOCAL, f'judge_{name}'), k=K, max_len=MAX_LEN, chunk_words=CHUNK_WORDS, device='cpu'))
    g8 = gw_scores(j, Q300); g = GW_CPU[name]
    thr = THR[name][DEFAULT['kind']][DEFAULT['threshold']]
    pr, ag = float(np.corrcoef(g, g8)[0, 1]), agree(g, g8, thr)
    F1_3[name] = bool(pr >= 0.99 and ag >= 0.98)
    I8.append({'model': name, 'pearson': pr, 'agree': ag, 'pass': F1_3[name]})
    print(f"=== F1-3 int8 [{name}]: Pearson {pr:.4f}, 판정 일치 {ag:.3f} → {'충족 (int8 허용)' if F1_3[name] else '미달 (fp32만)'} ===")
    del j; gc.collect()

LAT = []
N_GPU, N_CPU = 200, 60
for name in NAMES:
    d = os.path.join(LOCAL, f'judge_{name}')
    if DEVICE == 'cuda':
        j = StudentJudge(d, k=K, max_len=MAX_LEN, chunk_words=CHUNK_WORDS, device='cuda')
        p50, p95, pm = bench(j, N_GPU)
        LAT.append({'model': name, 'config': f'gpu:{torch.cuda.get_device_name(0)}', 'p50': p50, 'p95': p95, 'prepare_ms': pm})
        del j; gc.collect(); torch.cuda.empty_cache()
    for nt in sorted({1, 2, 4, NT}):
        torch.set_num_threads(nt)
        for prec in ['fp32', 'int8']:
            if prec == 'int8' and not F1_3[name]: continue
            j = StudentJudge(d, k=K, max_len=MAX_LEN, chunk_words=CHUNK_WORDS, device='cpu')
            if prec == 'int8': quantize(j)
            p50, p95, pm = bench(j, N_CPU)
            LAT.append({'model': name, 'config': f'cpu:{nt}t:{prec}', 'p50': p50, 'p95': p95, 'prepare_ms': pm})
            print(f'  [{name}] cpu {nt}스레드 {prec}: p50 {p50:.0f} / p95 {p95:.0f} ms')
            del j; gc.collect()
torch.set_num_threads(NT)

BUDGET = 8.5          # ms/토큰 (L1 Haiku 평균 속도 환산 4토큰 ≈ 34ms)
def min_stride(p95):
    return next((s for s in [4, 8, 16] if p95 <= s * BUDGET), np.nan)
LAT = pd.DataFrame(LAT)
LAT['min_stride'] = [min_stride(v) for v in LAT['p95']]
print(LAT.round(1).to_string(index=False))

def recall_at(name, s):
    k = str(s) if s else 'end'
    row = dev_tab[(dev_tab.model == name) & (dev_tab.kind == k) & (dev_tab.thr == 'svc10_orig') & (dev_tab.eval_labels == 'orig')].iloc[0]
    return k, row['recall'], row['delay_med']

def choose(cfg_prefix):
    # 사전 규칙: base(s ≤ 8) → mini(s ≤ 8) → base 문장 끝만. F1-1·F1-2 미달 모델은 제외
    for name in NAMES:
        if not (F1_1[name] and F1_2[name]): continue
        c = LAT[(LAT.model == name) & LAT.config.str.startswith(cfg_prefix)]
        c = c[c.min_stride.notna() & (c.min_stride <= 8)]
        if len(c):
            best = c.sort_values(['min_stride', 'p95']).iloc[0]
            return {'model': name, 'config': best.config, 'stride_tokens': int(best.min_stride), 'p95_ms': float(best.p95)}
    return {'model': 'base', 'config': cfg_prefix, 'stride_tokens': None, 'p95_ms': None}   # 문장 끝만

DEPLOY = {}
if DEVICE == 'cuda': DEPLOY['gpu'] = choose('gpu')
DEPLOY['cpu'] = choose(f'cpu:{NT}t')
for where, dcfg in DEPLOY.items():
    k, rc, dl = recall_at(dcfg['model'], dcfg['stride_tokens'])
    dcfg.update({'devL_svc10_recall_orig': float(rc), 'devL_delay_med': float(dl)})
    print(f"=== 배포 구성 [{where}]: {dcfg['model']} / {dcfg['config']} / 재판정 간격 {dcfg['stride_tokens'] or '문장 끝만'}토큰 "
          f"(p95 {dcfg['p95_ms'] or float('nan'):.0f}ms) → dev-L 서비스 10% 재현율 {rc:.3f}, 지연 {dl:.0f}토큰 ===")

## 13. 저장 (판정기 폴더 + 요약) 과 zip 다운로드

In [ ]:
for name in NAMES:
    d = os.path.join(LOCAL, f'judge_{name}')
    cfg = {'name': f'judge_{name}', 'backbone': BACKBONE_OF[name], 'seed': 42,
           'trained_on': 'RAGTruth train gold-large 3,000 responses, sentence-local labels + R4b Sonnet re-judged labels (C1-style)',
           'embedder': 'sentence-transformers/all-MiniLM-L6-v2', 'k': K, 'max_len': MAX_LEN, 'chunk_words': CHUNK_WORDS,
           'stride_unit': 'tiktoken cl100k_base tokens, counted from sentence start (sentence end always judged)',
           'default': DEFAULT, 'threshold': THR[name][DEFAULT['kind']][DEFAULT['threshold']], 'thresholds': THR[name],
           'score': 'logit (higher = hallucinated)', 'weights_dtype': 'float16 (load as float32)',
           'f1': {'reproduce': F1_1[name], 'gateway_parity': F1_2[name], 'int8_ok': F1_3[name]}}
    json.dump(cfg, open(os.path.join(d, 'judge_config.json'), 'w'), indent=1, ensure_ascii=False)

summ = {'F1_1': F1_1, 'F1_2': F1_2, 'F1_3': F1_3, 'deploy': DEPLOY, 'default': DEFAULT, 'drive_weights_ok': DRIVE_OK,
        'reproduce': rep.to_dict('records'), 'parity': PAR.to_dict('records'), 'int8': I8, 'thresholds': THR}
json.dump(summ, open(os.path.join(LOCAL, 'f1_summary.json'), 'w'), default=float, ensure_ascii=False, indent=1)
dev_tab.to_csv(os.path.join(LOCAL, 'f1_devL.csv'), index=False)
LAT.to_csv(os.path.join(LOCAL, 'f1_latency.csv'), index=False)
for name in NAMES:
    to_drive(name)                                   # judge_config·고정 예제 포함 다시 복사
for f in ['f1_summary.json', 'f1_devL.csv', 'f1_latency.csv']:
    try: shutil.copy(os.path.join(LOCAL, f), os.path.join(FINAL_DIR, f))
    except OSError as e: print('드라이브 저장 실패', f, e)

zp = shutil.make_archive('/content/f1_final_judges', 'zip', LOCAL)
print(f'zip {os.path.getsize(zp) / 2**20:.0f}MB:', zp)
try:
    shutil.copy(zp, os.path.join(FINAL_DIR, 'f1_final_judges.zip'))
except OSError as e:
    print('zip 드라이브 저장 실패', e)
from google.colab import files
files.download(zp)
print(json.dumps({k: summ[k] for k in ['F1_1', 'F1_2', 'F1_3', 'deploy', 'drive_weights_ok']}, ensure_ascii=False, indent=1, default=float))

## 14. (선택) T4 등 다른 GPU에서 판정 지연만 재기

배포 후보 GPU(T4 등)에서 런타임을 새로 열고 **이 셀 하나만** 실행한다. 드라이브 `final/judge_*`와 고정 예제를 쓴다.

In [ ]:
JUDGE_SRC = '"""문장 판정기.\n\nS2/S2-b와 같은 방식:\n  근거 문서 → 40단어 조각 → 조각 임베딩 (요청당 1회, prepare)\n  문장 → 임베딩 → 유사한 조각 top-k (원문 순서로 정렬) → 교차 인코더 → logit (클수록 환각)\n"""\nimport threading\nimport time\nfrom dataclasses import dataclass\n\nimport torch\n\n\n@dataclass\nclass Verdict:\n    score: float          # logit, 클수록 환각\n    hallucinated: bool\n    judge_ms: float       # 검색 + 판정 시간\n\n\n@dataclass\nclass RefIndex:\n    chunks: list\n    emb: object           # torch.Tensor [n_chunks, d]\n    prepare_ms: float\n\n\ndef chunk_text(text, n):\n    w = text.split()\n    return [\' \'.join(w[i:i + n]) for i in range(0, max(len(w), 1), n)] or [text]\n\n\nclass StudentJudge:\n    def __init__(self, model_dir, embedder=\'sentence-transformers/all-MiniLM-L6-v2\', k=8, max_len=512,\n                 chunk_words=40, threshold=0.0, device=\'\'):\n        from transformers import AutoModelForSequenceClassification, AutoTokenizer\n        self.device = device or (\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        self.tok = AutoTokenizer.from_pretrained(model_dir)\n        # 가중치가 fp16으로 저장돼 있어도 fp32로 올린다 (CPU 판정, 노트북 점수와 일치). GPU에서는 autocast가 반정밀도로 돌린다.\n        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir).float().to(self.device).eval()\n        if isinstance(embedder, str):\n            from sentence_transformers import SentenceTransformer\n            embedder = SentenceTransformer(embedder, device=self.device)\n        self.embedder = embedder\n        self.k, self.max_len, self.chunk_words, self.threshold = k, max_len, chunk_words, threshold\n        self._lock = threading.Lock()     # 모델 1개를 여러 요청이 공유하므로 직렬화\n\n    def _embed(self, texts):\n        return self.embedder.encode(texts, convert_to_tensor=True, normalize_embeddings=True,\n                                    show_progress_bar=False)\n\n    def prepare(self, reference):\n        t0 = time.perf_counter()\n        chunks = chunk_text(reference, self.chunk_words)\n        with self._lock:\n            emb = self._embed(chunks)\n        return RefIndex(chunks, emb, (time.perf_counter() - t0) * 1000)\n\n    @torch.no_grad()\n    def judge(self, ref, sentence):\n        t0 = time.perf_counter()\n        with self._lock:\n            e = self._embed([sentence])[0]\n            top = torch.topk(ref.emb @ e, min(self.k, ref.emb.shape[0])).indices.tolist()\n            ctx = \' ... \'.join(ref.chunks[i] for i in sorted(top))\n            enc = self.tok([ctx], [sentence], truncation=\'only_first\', max_length=self.max_len,\n                           return_tensors=\'pt\').to(self.device)\n            with torch.autocast(self.device, enabled=(self.device == \'cuda\')):\n                score = float(self.model(**enc).logits.squeeze().float().cpu())\n            if self.device == \'cuda\':\n                torch.cuda.synchronize()\n        return Verdict(score, score >= self.threshold, (time.perf_counter() - t0) * 1000)\n\n\nclass LexicalJudge:\n    """모델 없이 파이프라인을 시험하기 위한 대체 판정기 (테스트·데모 전용).\n    근거에 없는 단어 비율을 점수로 쓴다. 판별력은 없다고 봐야 한다."""\n\n    def __init__(self, threshold=0.5, delay_ms=0.0):\n        self.threshold, self.delay_ms = threshold, delay_ms\n\n    def prepare(self, reference):\n        return RefIndex([reference], {w.lower().strip(\'.,\') for w in reference.split()}, 0.0)\n\n    def judge(self, ref, sentence):\n        t0 = time.perf_counter()\n        if self.delay_ms:\n            time.sleep(self.delay_ms / 1000)\n        words = [w.lower().strip(\'.,\') for w in sentence.split() if w.strip(\'.,\')]\n        score = sum(w not in ref.emb for w in words) / max(len(words), 1)\n        return Verdict(score, score >= self.threshold, (time.perf_counter() - t0) * 1000)\n'
import os, json, numpy as np, torch, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'sentence-transformers'], check=True)
from google.colab import drive
drive.mount('/content/drive')
open('hallugate_judge.py', 'w').write(JUDGE_SRC); sys.path.insert(0, '.')
from hallugate_judge import StudentJudge
FD = '/content/drive/MyDrive/selfproject/final'
for name in ['base', 'mini']:
    d = os.path.join(FD, f'judge_{name}')
    fx = json.load(open(os.path.join(d, 'parity_fixture.json')))
    j = StudentJudge(d, k=fx['k'], max_len=fx['max_len'], chunk_words=fx['chunk_words'], device='cuda')
    refs, ms, diff = {}, [], []
    for it in fx['items']:
        if it['reference'] not in refs: refs[it['reference']] = j.prepare(it['reference'])
    for it in fx['items'][:5]:
        j.judge(refs[it['reference']], 'warm up.')
    for _ in range(4):
        for it in fx['items']:
            v = j.judge(refs[it['reference']], it['text'])
            ms.append(v.judge_ms); diff.append(abs(v.score - it['score_cpu_fp32']))
    print(f"{name} @ {torch.cuda.get_device_name(0)}: p50 {np.percentile(ms, 50):.1f} / p95 {np.percentile(ms, 95):.1f} ms | "
          f"CPU fp32 점수와 최대 차이 {max(diff):.3f}")